In [1]:
import numpy as np
import pandas as pd

In [2]:
hospital = pd.read_csv(r"C:\Users\manju\OneDrive\Desktop\Documents\Data Analytics\DA Semester 4\Capstone 2\Supplementary Data\hospital_kpi_ready.csv")
hospital.head().T

,0,1,2,3,4
year,2011,2011,2011,2011,2011
state_code,AL,MT,AL,AL,FL
provider_type,General Short-Term (includes CAH),General Short-Term (includes CAH),General Short-Term (includes CAH),Rehabilitation Hospital,Rehabilitation Hospital
rural_versus_urban,Rural,Rural,Urban,Urban,Urban
ccn_facility_type,Short-Term Hospital,Critical Access Hospital,Short-Term Hospital,Rehabilitation Hospital,Rehabilitation Hospital
number_of_beds,114.0,25.0,46.0,100.0,70.0
total_bed_days_available,41610.0,9125.0,16790.0,36500.0,25550.0
occupancy_rate,0.472026,0.131068,0.159678,0.887068,0.663836
total_discharges__v___xviii___xix___unknown_,5283.0,155.0,892.0,2390.0,1370.0
total_days__v___xviii___xix___unknown_,19641.0,1196.0,2681.0,32378.0,16961.0


In [3]:
rucc_raw = pd.read_csv(r"C:\Users\manju\OneDrive\Desktop\Documents\Data Analytics\DA Semester 4\Capstone 2\Supplementary Data\rural_urban_codes.csv",encoding='latin1')
rucc_raw.head()

,FIPS,State,County_Name,Attribute,Value
0,1001,AL,Autauga County,Population_2020,58805
1,1001,AL,Autauga County,RUCC_2023,2
2,1001,AL,Autauga County,Description,"Metro - Counties in metro areas of 250,000 to ..."
3,1003,AL,Baldwin County,Population_2020,231767
4,1003,AL,Baldwin County,RUCC_2023,3


In [ ]:
# Function to clean county names
import re

def clean_county(x):
    x = str(x).upper().strip()
    x = re.sub(r"\s+(COUNTY|MUNICIPIO|PARISH|BOROUGH|CENSUS AREA|ISLAND|CITY)$", "", x)
    x = x.replace("SAINT ", "ST ")
    x = re.sub(r"[^\w\s]", "", x)
    return x.strip()


In [ ]:
# CMS
hospital["county_key"] = hospital["county"].apply(clean_county)
hospital["state_key"] = hospital["state_code"].str.upper()

# RUCC 
rucc = rucc_raw.copy()
rucc.columns = rucc.columns.str.strip()

# Filter for RUCC 2023 attribute and valid numeric values
rucc = rucc[rucc["Attribute"] == "RUCC_2023"]
rucc = rucc[rucc["Value"].astype(str).str.isdigit()]

# Clean county and state keys
rucc["county_key"] = rucc["County_Name"].apply(clean_county)
rucc["state_key"] = rucc["State"].str.upper()
# Create RUCC code and rural/urban classification
rucc["rucc_code"] = rucc["Value"].astype(int)
rucc["rural_urban"] = rucc["rucc_code"].apply(
    lambda x: "Urban" if x <= 3 else "Rural"
)
# Select relevant columns
rucc = rucc[["state_key", "county_key", "rucc_code", "rural_urban"]]


In [ ]:
# Check overlap of county keys in Alabama
cms_al = set(hospital[hospital["state_key"]=="AL"]["county_key"])
rucc_al = set(rucc[rucc["state_key"]=="AL"]["county_key"])
cms_al & rucc_al


{'AUTAUGA',
 'BALDWIN',
 'BARBOUR',
 'BIBB',
 'BLOUNT',
 'BULLOCK',
 'BUTLER',
 'CALHOUN',
 'CHAMBERS',
 'CHEROKEE',
 'CHILTON',
 'CLARKE',
 'CLAY',
 'COFFEE',
 'COLBERT',
 'CONECUH',
 'COVINGTON',
 'CRENSHAW',
 'CULLMAN',
 'DALE',
 'DALLAS',
 'DEKALB',
 'ELMORE',
 'ESCAMBIA',
 'ETOWAH',
 'FAYETTE',
 'FRANKLIN',
 'GENEVA',
 'GREENE',
 'HALE',
 'HOUSTON',
 'JACKSON',
 'JEFFERSON',
 'LAUDERDALE',
 'LAWRENCE',
 'LEE',
 'LIMESTONE',
 'MADISON',
 'MARENGO',
 'MARION',
 'MARSHALL',
 'MOBILE',
 'MONROE',
 'MONTGOMERY',
 'MORGAN',
 'PICKENS',
 'PIKE',
 'RANDOLPH',
 'RUSSELL',
 'SHELBY',
 'ST CLAIR',
 'SUMTER',
 'TALLADEGA',
 'TALLAPOOSA',
 'TUSCALOOSA',
 'WALKER',
 'WASHINGTON',
 'WILCOX',
 'WINSTON'}

In [ ]:
# Merge RUCC data into hospital dataset
hospital = hospital.merge(
    rucc,
    on=["state_key", "county_key"],
    how="left"
).drop(columns=["state_key", "county_key"])


In [13]:
# Dropping rural_versus_urban column as not needed
hospital = hospital.drop(columns=["rural_versus_urban"])
hospital.head().T

,0,1,2,3,4
year,2011,2011,2011,2011,2011
state_code,AL,MT,AL,AL,FL
provider_type,General Short-Term (includes CAH),General Short-Term (includes CAH),General Short-Term (includes CAH),Rehabilitation Hospital,Rehabilitation Hospital
ccn_facility_type,Short-Term Hospital,Critical Access Hospital,Short-Term Hospital,Rehabilitation Hospital,Rehabilitation Hospital
number_of_beds,114.0,25.0,46.0,100.0,70.0
total_bed_days_available,41610.0,9125.0,16790.0,36500.0,25550.0
occupancy_rate,0.472026,0.131068,0.159678,0.887068,0.663836
total_discharges__v___xviii___xix___unknown_,5283.0,155.0,892.0,2390.0,1370.0
total_days__v___xviii___xix___unknown_,19641.0,1196.0,2681.0,32378.0,16961.0
fte___employees_on_payroll,598.72,70.56,72.65,297.77,154.6


In [14]:
# Saving the updated hospital dataset
hospital.to_csv(r"C:\Users\manju\OneDrive\Desktop\Documents\Data Analytics\DA Semester 4\Capstone 2\ML Modelling\hospital_kpi_ready_with_rucc.csv", index=False)

In [15]:
# Display final dataset sample
cms_rucc_data = pd.read_csv(r"C:\Users\manju\OneDrive\Desktop\Documents\Data Analytics\DA Semester 4\Capstone 2\ML Modelling\hospital_kpi_ready_with_rucc.csv")
cms_rucc_data.head().T

,0,1,2,3,4
year,2011,2011,2011,2011,2011
state_code,AL,MT,AL,AL,FL
provider_type,General Short-Term (includes CAH),General Short-Term (includes CAH),General Short-Term (includes CAH),Rehabilitation Hospital,Rehabilitation Hospital
ccn_facility_type,Short-Term Hospital,Critical Access Hospital,Short-Term Hospital,Rehabilitation Hospital,Rehabilitation Hospital
number_of_beds,114.0,25.0,46.0,100.0,70.0
total_bed_days_available,41610.0,9125.0,16790.0,36500.0,25550.0
occupancy_rate,0.472026,0.131068,0.159678,0.887068,0.663836
total_discharges__v___xviii___xix___unknown_,5283.0,155.0,892.0,2390.0,1370.0
total_days__v___xviii___xix___unknown_,19641.0,1196.0,2681.0,32378.0,16961.0
fte___employees_on_payroll,598.72,70.56,72.65,297.77,154.6


In [16]:
hospital["rucc_code"].value_counts().sort_index()
hospital["rural_urban"].value_counts()

rural_urban
Urban    45062
Rural    23022
Name: count, dtype: int64